In [55]:
import pandas as pd
import numpy as np
import glob


In [56]:
# Load all docking CSVs
csv_files = sorted(glob.glob("/content/ligand*.csv"))

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()   # fix whitespace
    df["Source_File"] = f.split("/")[-1]
    dfs.append(df)

dock = pd.concat(dfs, ignore_index=True)

# Standardize column names
dock = dock.rename(columns={
    "Binding Affinity": "Docking_Energy",
    "rmsd/lb": "RMSD_lb"
})

print("Docking data shape:", dock.shape)
dock.head()


Docking data shape: (66060, 5)


,Ligand,Docking_Energy,rmsd/ub,RMSD_lb,Source_File
0,199_uff_e=50.33_out,-3.9,0.000,0.000,ligand 1.csv
1,199_uff_e=50.33_out,-3.8,3.409,1.385,ligand 1.csv
2,199_uff_e=50.33_out,-3.8,3.108,1.854,ligand 1.csv
3,199_uff_e=50.33_out,-3.6,4.089,2.089,ligand 1.csv
4,199_uff_e=50.33_out,-3.6,3.196,2.186,ligand 1.csv


In [57]:
# Extract CID from ligand name (e.g. 161671_uff_e=...)
dock["PubChem_CID"] = dock["Ligand"].str.extract(r"^(\d+)").astype(int)

dock[["Ligand", "PubChem_CID"]].head(10)


,Ligand,PubChem_CID
0,199_uff_e=50.33_out,199
1,199_uff_e=50.33_out,199
2,199_uff_e=50.33_out,199
3,199_uff_e=50.33_out,199
4,199_uff_e=50.33_out,199
5,199_uff_e=50.33_out,199
6,199_uff_e=50.33_out,199
7,199_uff_e=50.33_out,199
8,199_uff_e=50.33_out,199
9,264_uff_e=17.52_out,264


In [58]:
pubchem = pd.read_csv("/content/PubChem_compound.csv")

print("PubChem data shape:", pubchem.shape)
pubchem.head()


PubChem data shape: (2847, 38)


,Compound_CID,Name,Synonyms,Molecular_Weight,Molecular_Formula,Polar_Area,Complexity,XLogP,Heavy_Atom_Count,H-Bond_Donor_Count,...,Linked_PubChem_Patent_Count,Linked_PubChem_Patent_Family_Count,MeSH_Headings,Annotation_Content,Annotation_Type_Count,Linked_BioAssays,Create_Date,Data_Source,Data_Source_Category,Tagged_by_PubChem
0,199,Agmatine,agmatine|1-(4-Aminobutyl)guanidine|306-60-5|(4...,130.19,C5H14N4,90.4,85.0,-1.5,9,3,...,9328,2587,Agmatine,Biological Test Results|Interactions and Pathw...,14,357|410|411|444|445|446|447|448|450|451|526|53...,20040916,001Chemical|10X CHEM|1st Scientific|3WAY PHARM...,Chemical Vendors|Curation Efforts|Governmental...,NaN
1,264,Butyric Acid,butyric acid|butanoic acid|107-92-6|n-Butyric ...,88.11,C4H8O2,37.3,49.5,0.8,6,1,...,61178,25914,Butyric Acid,Biological Test Results|Interactions and Pathw...,18,155|157|161|165|167|175|192|200|206|212|220|25...,20040916,001Chemical|10X CHEM|1st Scientific|3WAY PHARM...,Chemical Vendors|Curation Efforts|Governmental...,D018377 - Neurotransmitter Agents > D018494 - ...
2,323,Coumarin,coumarin|91-64-5|2H-Chromen-2-one|2H-1-Benzopy...,146.14,C9H6O2,26.3,196.0,1.4,11,0,...,121878,41804,NaN,Biological Test Results|Interactions and Pathw...,18,1|3|7|9|11|13|15|17|19|21|23|25|27|29|31|33|35...,20040916,001Chemical|10X CHEM|3WAY PHARM INC|A&J Pharmt...,Chemical Vendors|Curation Efforts|Governmental...,C78275 - Agent Affecting Blood or Body Fluid >...
3,325,4-Isopropylbenzyl alcohol,4-isopropylbenzyl alcohol|536-60-7|Cumic alcoh...,150.22,C10H14O,20.2,101.0,2.3,11,1,...,8485,3024,NaN,Biological Test Results|Interactions and Pathw...,15,1201|485350|485395|493033|504770|540299|588519...,20050326,001Chemical|10X CHEM|3WAY PHARM INC|A&J Pharmt...,Chemical Vendors|Curation Efforts|Governmental...,NaN
4,326,Cuminaldehyde,4-isopropylbenzaldehyde|cuminaldehyde|122-03-2...,148.20,C10H12O,17.1,121.0,2.7,11,0,...,11179,3577,NaN,Biological Test Results|Interactions and Pathw...,14,155|157|161|165|167|175|1188|213388|293934|468...,20050326,001Chemical|10X CHEM|3WAY PHARM INC|A&J Pharmt...,Chemical Vendors|Curation Efforts|Governmental...,NaN


In [59]:
data = dock.merge(
    pubchem,
    left_on="PubChem_CID",
    right_on="Compound_CID",
    how="inner"
)

print("Merged data shape:", data.shape)


Merged data shape: (66060, 44)


In [60]:
# Absolute docking energy
data["Abs_Energy"] = data["Docking_Energy"].abs()

# Pose stability (lower RMSD = better)
epsilon = 1e-3
data["Pose_Stability"] = 1 / (data["RMSD_lb"] + epsilon)


In [61]:
data = data.rename(columns={
    "H-Bond_Donor_Count": "HBD",
    "H-Bond_Acceptor_Count": "HBA"
})


In [62]:
admet_cols = [
    "Molecular_Weight",
    "XLogP",
    "Polar_Area",
    "HBD",
    "HBA",
    "Rotatable_Bond_Count"
]

# Z-score normalization
for col in admet_cols:
    data[f"Z_{col}"] = (data[col] - data[col].mean()) / data[col].std()

# ADMET proxy score
data["ADMET_proxy"] = data[[f"Z_{c}" for c in admet_cols]].mean(axis=1)


In [63]:
data["Z_Energy"] = (data["Abs_Energy"] - data["Abs_Energy"].mean()) / data["Abs_Energy"].std()
data["Z_Stability"] = (data["Pose_Stability"] - data["Pose_Stability"].mean()) / data["Pose_Stability"].std()


In [64]:
w_energy = 0.5
w_stab   = 0.2
w_admet  = 0.3

data["Composite_Score"] = (
    w_energy * data["Z_Energy"] +
    w_stab   * data["Z_Stability"] +
    w_admet  * data["ADMET_proxy"]
)


In [65]:
# Rank all poses
ranked_all = data.sort_values("Composite_Score", ascending=False)

# Keep best pose per PubChem CID
best_per_ligand = (
    ranked_all
    .groupby("PubChem_CID", as_index=False)
    .first()
)

# Ligand-level rank
best_per_ligand["Ligand_Rank"] = (
    best_per_ligand["Composite_Score"]
    .rank(ascending=False, method="min")
)


In [66]:
top100 = (
    best_per_ligand
    .sort_values("Ligand_Rank")
    .head(100)
)

top100_table = top100[[
    "Ligand_Rank",
    "PubChem_CID",
    "Name",
    "Docking_Energy",
    "RMSD_lb",
    "Composite_Score"
]]

top100_table.head(10)


,Ligand_Rank,PubChem_CID,Name,Docking_Energy,RMSD_lb,Composite_Score
2134,1.0,5281711,Terminalin,-9.4,0.0,2.924165
1041,2.0,441913,"[(1'S,3'R,4'R,5'R,6'R,10'S,12'S,16'R,18'S,21'R...",-9.3,0.0,2.712073
2370,3.0,11227154,Pelargonidin 3-O-(6-caffeoyl-beta-D-glucoside),-8.7,0.0,2.681790
1752,4.0,5271805,Ginkgetin,-9.2,0.0,2.679796
2367,5.0,11157896,Delphinidin 3-O-(6-caffeoyl-beta-D-glucoside),-8.4,0.0,2.667882
1019,6.0,441862,Labriformin,-9.4,0.0,2.640913
2042,7.0,5281600,Amentoflavone,-9.1,0.0,2.631565
2441,8.0,15922818,Delphinidin-3-O-(6-p-coumaroyl)glucoside,-8.4,0.0,2.592436
2119,9.0,5281694,Robustaflavone,-9.0,0.0,2.576631
1315,10.0,442428,Naringin,-8.8,0.0,2.570275


In [67]:
top100_table.to_csv("Top100_Ligands_Composite_OptionA.csv", index=False)

from google.colab import files
files.download("Top100_Ligands_Composite_OptionA.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>